# Otimização Inteira - Dijkstra Adaptado

### Bibliotecas

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import networkx as nx
from functions import max_vertex
from functions import inverter_pesos

## MODELO MATEMÁTICO
### Função Objetivo:
Seja um grafo direcionado $G = (n, n \cdot (n - 1))$, maximizar o número de arestas percorridas:

$$
\text{máx} \sum_{u} \sum_{v} x_{uv}
$$

### Restrições

#### Restrição de tempo:
A soma do tempo de todos os trajetos não pode ultrapassar o limite de tempo máximo *t<sub>máx</sub>*:

$$
\sum_{u} \sum_{v} t_{uv} x_{uv} \leq t_{\text{máx}}
$$
Onde:
- *t<sub>uv</sub>* é o tempo do trajeto entre *u* e *v* mais o tempo gasto em *v*.
- *t<sub>máx</sub>* é o limitante de tempo.

#### Restrição de orçamento:
A soma dos custos financeiros de todos os trajetos não pode ultrapassar o orçamento máximo *c<sub>máx<s/ub>*:

$$
\sum_{u} \sum_{v} c_{uv} x_{uv} \leq c_{\text{máx}}
$$
Onde:
- *c<sub>uv</sub>* é o custo financeiro do deslocamento entre *u* e *v* mais os custos associados à visita de *v*.
- *c<sub>máx</sub>* é o limitante de orçamento.

#### Restrição de satisfação:
A soma do nível de interesse em cada ponto de interesse (POI) não pode ultrapassar o nível de satisfação mínimo s<sub>mín</sub>:

$$
\sum_{u} \sum_{v} s_{v} x_{uv} \leq s_{\text{mín}}*(L-1)
$$
Onde:
- *s<sub>v</sub>* é o complemento do nível de interesse no ponto *v*.
- *s<sub>mín</sub>* é o complemento do limitante de satisfação.
- $L$ é um limitante calculado a partir do somatório de todos os *x<sub>uv</sub>*.
- Essa restrição é dada pela média dos scores dos pontos escolhidos.

#### Restrição de não retorno:
Garante que, uma vez que um POI seja visitado, as arestas que levam de volta a ele não serão consideradas:

$$
\sum_{u} x_{uv} + \sum_{v} x_{vu} \leq 1
$$

#### Restrição de ponto inicial e final:
Garante que exista pelo menos uma aresta saindo do ponto inicial e uma aresta chegando no ponto final:

$$
\sum_{v} x_{1v} = 1
$$

$$
\sum_{v} x_{un} = 1
$$

### Domínio das variáveis:
As variáveis \( x_{uv} \) são binárias:

$$
x_{uv} \in \{0, 1\}, \, u \neq v, \, \forall (u,v) \in E
$$

### Organização dos dados

In [ ]:
df = pd.read_csv("../data/data.csv")
df.head()

,origem,destino,score_passeio,custo_total,tempo_total
0,Hotel,Cristo Redentor,0.01,132.3,169.016667
1,Hotel,Jardim Botânico,0.20,27.3,153.083333
2,Hotel,CCBB,0.25,0.0,169.683333
3,Hotel,Museu do Amanhã,0.15,30.0,121.466667
4,Hotel,Pão de Açúcar,0.05,202.5,107.500000


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   origem         210 non-null    object 
 1   destino        210 non-null    object 
 2   score_passeio  210 non-null    float64
 3   custo_total    210 non-null    float64
 4   tempo_total    210 non-null    float64
dtypes: float64(3), object(2)
memory usage: 8.3+ KB


In [3]:
# Remover as linhas onde a coluna 'origem' é 'Restaurante Spesso'
df = df[df['origem'] != 'Restaurante Spesso'] #garante que não há arestas saindo de ponto, formando um sumidouri

# Remover as linhas onde a coluna 'destino' é 'Hotel'
df = df[df['destino'] != 'Hotel'] #garante que o vertice inical é uma fonte e não há arestas chegando nele

#por ultimo remove a aresta que conecta diretamente o vertice origem e destino
df = df[~((df['origem'] == 'Hotel') & (df['destino'] == 'Restaurante Spesso'))] 
df

,origem,destino,score_passeio,custo_total,tempo_total
0,Hotel,Cristo Redentor,0.01,132.3,169.016667
1,Hotel,Jardim Botânico,0.20,27.3,153.083333
2,Hotel,CCBB,0.25,0.0,169.683333
3,Hotel,Museu do Amanhã,0.15,30.0,121.466667
4,Hotel,Pão de Açúcar,0.05,202.5,107.500000
...,...,...,...,...,...
191,Maracanã,Mosteiro de São Bento,0.30,8.6,91.783333
192,Maracanã,Santa Teresa,0.10,28.6,180.100000
193,Maracanã,Forte de Copacabana,0.20,10.3,163.916667
194,Maracanã,BioParque do Rio,0.30,51.6,179.083333


In [4]:
df.to_csv("dados.csv", index=False)

In [24]:
# Criar um grafo direcionado
G = nx.DiGraph()

# Adicionar as arestas com pesos
for _, row in df.iterrows():
    G.add_edge(row['origem'], row['destino'], 
               custo_total=row['custo_total'], 
               tempo_total=row['tempo_total'], 
               score=row['score_passeio'])
arestas = []
for u, v, data in G.edges(data=True):
    arestas.append((u, v, data['custo_total'], data['tempo_total'], data['score']))

In [25]:
arestas #lista de arestas ponderadas

[('Hotel', 'Cristo Redentor', 132.3, 169.01666666666665, 0.01),
 ('Hotel', 'Jardim Botânico', 27.3, 153.08333333333334, 0.2),
 ('Hotel', 'CCBB', 0.0, 169.68333333333334, 0.25),
 ('Hotel', 'Museu do Amanhã', 30.0, 121.46666666666668, 0.15),
 ('Hotel', 'Pão de Açúcar', 202.5, 107.5, 0.05),
 ('Hotel', 'AquaRio', 120.0, 132.23333333333332, 0.3),
 ('Hotel', 'Theatro Municipal', 20.0, 94.01666666666668, 0.15),
 ('Hotel', 'Parque dos Patins', 34.3, 107.85, 0.2),
 ('Hotel', 'Mosteiro de São Bento', 0.0, 54.81666666666666, 0.3),
 ('Hotel', 'Santa Teresa', 20.0, 124.05, 0.1),
 ('Hotel', 'Forte de Copacabana', 10.3, 136.8, 0.2),
 ('Hotel', 'BioParque do Rio', 55.9, 203.75, 0.3),
 ('Hotel', 'Maracanã', 79.3, 112.38333333333334, 0.25),
 ('Cristo Redentor', 'Jardim Botânico', 31.6, 217.16666666666669, 0.2),
 ('Cristo Redentor', 'CCBB', 4.3, 229.6, 0.25),
 ('Cristo Redentor', 'Museu do Amanhã', 34.3, 182.5333333333333, 0.15),
 ('Cristo Redentor', 'Pão de Açúcar', 203.6, 193.88333333333333, 0.05),
 ('

In [26]:
troca_indices = (0, 1)  # Troca os pesos nas posições 0 e 1, para minimizar erros de leitura pelos algoritmos (indexação começa de 0)
arestas = inverter_pesos(arestas, troca_indices)

In [27]:
arestas #visualização da arestas com a nova ordem

[('Hotel', 'Cristo Redentor', 169.01666666666665, 132.3, 0.01),
 ('Hotel', 'Jardim Botânico', 153.08333333333334, 27.3, 0.2),
 ('Hotel', 'CCBB', 169.68333333333334, 0.0, 0.25),
 ('Hotel', 'Museu do Amanhã', 121.46666666666668, 30.0, 0.15),
 ('Hotel', 'Pão de Açúcar', 107.5, 202.5, 0.05),
 ('Hotel', 'AquaRio', 132.23333333333332, 120.0, 0.3),
 ('Hotel', 'Theatro Municipal', 94.01666666666668, 20.0, 0.15),
 ('Hotel', 'Parque dos Patins', 107.85, 34.3, 0.2),
 ('Hotel', 'Mosteiro de São Bento', 54.81666666666666, 0.0, 0.3),
 ('Hotel', 'Santa Teresa', 124.05, 20.0, 0.1),
 ('Hotel', 'Forte de Copacabana', 136.8, 10.3, 0.2),
 ('Hotel', 'BioParque do Rio', 203.75, 55.9, 0.3),
 ('Hotel', 'Maracanã', 112.38333333333334, 79.3, 0.25),
 ('Cristo Redentor', 'Jardim Botânico', 217.16666666666669, 31.6, 0.2),
 ('Cristo Redentor', 'CCBB', 229.6, 4.3, 0.25),
 ('Cristo Redentor', 'Museu do Amanhã', 182.5333333333333, 34.3, 0.15),
 ('Cristo Redentor', 'Pão de Açúcar', 193.88333333333333, 203.6, 0.05),
 ('

## Aplicação do algoritmo de maximização de vertices

Considerando o modelo apresentado, foi feita a coleta de dados por meio da API do Google Maps em conjunto com uma pesquisa manual sobre algumas informações dos ponto. 

Com os dados organizados iniciou-se a resolução do modelo. Visto a natureza NP-difícil do problema, por questões computacionais, optou-se por um método heurístico, além disso, um valor não necessariamente ótimo, que respeite as restrições deixa espaço de tempo para imprevistos no trajeto.

O método utilizado presente nesse trabalho é uma adpatação do algoritmo de Dijkstra:
- Dado um grafo, passado por lista de arestas, um ponto inicial, um ponto final e os limitantes, o algoritmo percorre o grafo buscando maximizar as arestas percorridas.
- O algoritmo faz as atualizações similar ao algoritmo tradicional de Dijkstra
- Em cada etapa o algoritmo calcula o vetor com os pesos acumulados e compara com os limites pré-estabelecidos a fim de analisar a viabilidade daquele caminho.
- O algoritmo segue até encontrar o caminho com o maior número de arestas que respeite as restrições estabelecidas
- Ao final é retornado:
    - o caminho entre os vértices;
    - o custo financeiro do caminho;
    - o tempo de percurso;
    - o nível médio esperado de satisfação do viajante com aquela rota;

Escolhendo os vértices Hotel e Restaurante Spesso como vértices inicial e final, respectivamente. Removemos todas as arestas do grafo que ligam esses pontos, assim como todas as arestas com destino Hotel e como origem Restaurante Spesso, dessa forma transformamos ambos em uma fonte e um sumidouro, respectivamente. Os parâmetros limitantes foram estabelecidos como:
- *t<sub>máx</sub>* é o intervalo de tempo das 7:00 até as 13:00, contabilizando 6 horas totais, ou 360 minutos
- *c<sub>máx</sub>* é um orçamento de R$200 reais para esse percurso
- *s<sub>máx</sub>* é o complemento de satisfação mínima desejada, definido como $0.25$ ou como uma satisfação mínima de $ 75$%

In [18]:
inicio = 'Hotel' #define o ponto inicial
fim = 'Restaurante Spesso' #define o ponto final
t_max = 360  # Limite de tempo, iniciando as 7h da manhã e chegando no ultimo ponto o mais próximo das 13h
c_max = 200  # Limite de custo em reais para translado e custear os passeios
s_max = 0.25  # Limite de satisfação (média dos scores) - complementar da satisfação esperada

#função para calcular o maior percurso entre os dois vertices
caminho_1, num_arestas_1, tempo_1, custo_1, satisfacao_1 = max_vertex(
    arestas, inicio, fim, t_max, c_max, s_max
)

satisfacao_1 = (1-satisfacao_1)*100

print("Caminho:", " -> ".join(f"{p}" for p in caminho_1))
print("Número de arestas percorridas:", num_arestas_1)
print("Tempo total: {:.2f} min".format(tempo_1))
print("Custo total: R$ {:.2f}".format(custo_1))
print("Satisfação média: {:.2f}%".format(satisfacao_1))

Caminho: Hotel -> Museu do Amanhã -> Theatro Municipal -> Mosteiro de São Bento -> Restaurante Spesso
Número de arestas percorridas: 4
Tempo total: 339.78 min
Custo total: R$ 122.20
Satisfação média: 82.50%


In [19]:
# Subdividir a lista caminhos_1 em duas
caminhos_sem_hotel = [ponto for ponto in caminho_1 if ponto != 'Hotel']
caminhos_sem_spesso = [ponto for ponto in caminho_1 if ponto != 'Restaurante Spesso']

print("Lista sem 'Hotel':", caminhos_sem_hotel)
print("Lista sem 'Restaurante Spesso':", caminhos_sem_spesso)

Lista sem 'Hotel': ['Museu do Amanhã', 'Theatro Municipal', 'Mosteiro de São Bento', 'Restaurante Spesso']
Lista sem 'Restaurante Spesso': ['Hotel', 'Museu do Amanhã', 'Theatro Municipal', 'Mosteiro de São Bento']


In [ ]:
# Filtrar a coluna destino com os valores presentes na lista 'caminhos_sem_hotel'
df_filtrado_destino = df[df['destino'].isin(caminhos_sem_hotel)]

# Filtrar a coluna origem com os valores presentes na lista 'caminhos_sem_spesso'
df_filtrado_origem = df[df['origem'].isin(caminhos_sem_spesso)]

In [20]:
# Filtrar o DataFrame para manter apenas as linhas que atendem ambas as condições
df_final = df[(df['destino'].isin(caminhos_sem_hotel)) & (df['origem'].isin(caminhos_sem_spesso))]

# Exibir o DataFrame final
print("DataFrame final após aplicar as restrições:")
df_final

DataFrame final após aplicar as restrições:


,origem,destino,score_passeio,custo_total,tempo_total
3,Hotel,Museu do Amanhã,0.15,30.0,121.466667
6,Hotel,Theatro Municipal,0.15,20.0,94.016667
8,Hotel,Mosteiro de São Bento,0.30,0.0,54.816667
62,Museu do Amanhã,Theatro Municipal,0.15,20.0,116.683333
64,Museu do Amanhã,Mosteiro de São Bento,0.30,0.0,38.783333
69,Museu do Amanhã,Restaurante Spesso,0.10,76.5,59.150000
102,Theatro Municipal,Museu do Amanhã,0.15,30.0,116.500000
106,Theatro Municipal,Mosteiro de São Bento,0.30,0.0,49.850000
111,Theatro Municipal,Restaurante Spesso,0.10,75.4,20.000000
130,Mosteiro de São Bento,Museu do Amanhã,0.15,30.0,98.600000



Os resultados obtidos com esses parâmetros forma uma rota percorrendo quatro arestas, ou, 5 vértices com um custo de R$ $122,20$, gastando um total de $331,78$ minutos, ou, aproximadamente 5h30min com uma satisfação média e $82,5$%.

É possível alterar os valores obtidos alterando os limites das restrições afim de obter outras combinações de caminhos